In [7]:
import os
import json
import polars as pl
from ucimlrepo import fetch_ucirepo

import kagglehub
from kagglehub import KaggleDatasetAdapter
# from kaggle.api.kaggle_api_extended import KaggleApi

import mixtabank
from mixtabank.src.utils import get_df_info_pl

In [9]:
# mixtabank.uci_dict

In [ ]:
# # read a JSON file into a dict:
# with open('data_src_dict/UCI.JSON') as f:
#     uci_dict = json.load(f)

In [6]:
# datasets_path = "/Users/leec/Documents/datasets/"

### Petfinder_Tab

In [14]:
dataset_path = "datuman/petfinder-adoption-tabular-only-tables"
file_name = "petfinder_tab_processed.csv"

# Load the file directly into a Polars DataFrame
df = kagglehub.dataset_load(KaggleDatasetAdapter.POLARS, dataset_path, file_name).collect()
df

Type,Gender,Breed1,Color1,Color2,State,Age,MaturitySize,FurLength,Vaccinated,Dewormed,Sterilized,Health,Quantity,Fee,VideoAmt,PhotoAmt,is_adopted
str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""Cat""","""Male""","""Tabby""","""Black""","""White""","""Selangor""",3,1,1,2,2,2,1,1,100,0,1,1
"""Cat""","""Male""","""Domestic Medium Hair""","""Black""","""Brown""","""Kuala Lumpur""",1,2,2,3,3,3,1,1,0,0,2,1
"""Dog""","""Male""","""Mixed Breed""","""Brown""","""White""","""Selangor""",1,2,2,1,1,2,1,1,0,0,7,1
"""Dog""","""Female""","""Mixed Breed""","""Black""","""Brown""","""Kuala Lumpur""",4,2,1,1,1,2,1,1,150,0,8,1
"""Dog""","""Male""","""Mixed Breed""","""Black""","""NoColor""","""Selangor""",1,2,1,2,2,2,1,1,0,0,3,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Cat""","""Mixed""","""Domestic Short Hair""","""Black""","""NoColor""","""Selangor""",2,2,2,2,2,2,1,4,0,0,3,1
"""Cat""","""Mixed""","""Domestic Medium Hair""","""Black""","""Yellow""","""Selangor""",60,2,2,1,1,1,1,2,0,0,3,0
"""Cat""","""Mixed""","""Domestic Medium Hair""","""Cream""","""Gray""","""Selangor""",2,3,2,2,1,3,1,5,30,0,5,1


In [8]:
dataset_name = "petfinder_tab"
prediction_task = "binary_classification"  # ,regression,  multiclass_classification, binary_classification
target_col_name = "is_adopted"


petfinder_dtypes_final = {
    "Type": pl.Utf8,
    "Gender": pl.Utf8,
    "Breed1": pl.Utf8,
    "Color1": pl.Utf8,
    "Color2": pl.Utf8,
    "State": pl.Utf8,
    "Age": pl.Int64,
    "MaturitySize": pl.Int64,
    "FurLength": pl.Int64,
    "Vaccinated": pl.Int64,
    "Dewormed": pl.Int64,
    "Sterilized": pl.Int64,
    "Health": pl.Int64,
    "Quantity": pl.Int64,
    "Fee": pl.Int64,
    "VideoAmt": pl.Int64,
    "PhotoAmt": pl.Int64,
    "is_adopted": pl.Utf8,
}

petfinder_dtypes_final.pop("is_adopted")
catCols = [k for k, v in petfinder_dtypes_final.items() if v == pl.Utf8]
numCols = [k for k, v in petfinder_dtypes_final.items() if v != pl.Utf8]

# Load and preprocess data
petfinder_data_path_train = os.path.join(datasets_path, dataset_name, "train.csv")
petfinder_data_path_test = os.path.join(datasets_path, dataset_name, "test.csv")
petfinder_color_path = os.path.join(datasets_path, dataset_name, "PetFinder-ColorLabels.csv")
petfinder_breed_path = os.path.join(datasets_path, dataset_name, "PetFinder-BreedLabels.csv")
petfinder_state_path = os.path.join(datasets_path, dataset_name, "PetFinder-StateLabels.csv")

dataset = pl.read_csv(petfinder_data_path_train)  # no nulls in this dataset
petfinder_color_df = pl.read_csv(petfinder_color_path)  # color lookup table
petfinder_breed_df = pl.read_csv(petfinder_breed_path)  # breed lookup table
petfinder_state_df = pl.read_csv(petfinder_state_path)  # state lookup table

# Join lookup tables to main dataset, and rename columns.
dataset = (
    dataset.join(petfinder_breed_df, left_on=["Breed1", "Type"], right_on=["BreedID", "Type"], how="left")
    .drop("Breed1")
    .rename({"BreedName": "Breed1"})
)
dataset = (
    dataset.join(petfinder_color_df, left_on="Color1", right_on="ColorID", how="left")
    .drop("Color1")
    .rename({"ColorName": "Color1"})
)
dataset = (
    dataset.join(petfinder_color_df, left_on="Color2", right_on="ColorID", how="left")
    .drop("Color2")
    .rename({"ColorName": "Color2"})
)
dataset = dataset.with_columns(pl.col("Color2").fill_null("NoColor"))
dataset = (
    dataset.join(petfinder_state_df, left_on="State", right_on="StateID", how="left")
    .drop("State")
    .rename({"StateName": "State"})
)

# Create target column and convert some columns to string type.
dataset = dataset.with_columns(pl.when(pl.col("AdoptionSpeed") == 4).then(0).otherwise(1).alias("is_adopted"))
dataset = dataset.with_columns(pl.when(pl.col("Type") == 1).then(pl.lit("Dog")).otherwise(pl.lit("Cat")).alias("Type"))
dataset = dataset.with_columns(
    pl.when(pl.col("Gender") == 1)
    .then(pl.lit("Male"))
    .when(pl.col("Gender") == 2)
    .then(pl.lit("Female"))
    .otherwise(pl.lit("Mixed"))
    .alias("Gender")
)

dataset = dataset.with_columns(pl.col("PhotoAmt").cast(pl.Int64))

# Drop un-used columns.
dataset = dataset.select(catCols + numCols + ["is_adopted"])

# dataset.write_csv(os.path.join(datasets_path,dataset_name,"petfinder_tab_processed.csv"))
# final shape: (14993, 18)

# dataset.null_count().sum()

In [9]:
print(f"Number of numerical columns: {len(numCols)}")
print(f"Number of categorical columns: {len(catCols)}")
print(f"Total cardinality: {sum([dataset[col].n_unique() for col in catCols])}")

Number of numerical columns: 11
Number of categorical columns: 6
Total cardinality: 208


In [4]:
df_info = get_df_info_pl(dataset)
# df_info

NameError: name 'dataset' is not defined

In [3]:
os.environ["KAGGLE_USERNAME"] = "datuman"
os.environ["KAGGLE_KEY"] = "87e8f21e324595550a4dced6a311b355"

api = KaggleApi()
api.authenticate()

In [ ]:
api.dataset_list

In [6]:
# datasets = api.dataset_list(search='home-credit-default-risk')
# print(datasets)

In [18]:


# # Download latest version
# path = kagglehub.dataset_download("megancrenshaw/home-credit-default-risk")

# print("Path to dataset files:", path)

In [5]:
dataset_path = "megancrenshaw/home-credit-default-risk"
file_name = "home-credit-default-risk/application_train.csv"

# Load the file directly into a Polars DataFrame
df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, dataset_path, file_name)#.collect()
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
df.dtypes

SK_ID_CURR                      int64
TARGET                          int64
NAME_CONTRACT_TYPE             object
CODE_GENDER                    object
FLAG_OWN_CAR                   object
                               ...   
AMT_REQ_CREDIT_BUREAU_DAY     float64
AMT_REQ_CREDIT_BUREAU_WEEK    float64
AMT_REQ_CREDIT_BUREAU_MON     float64
AMT_REQ_CREDIT_BUREAU_QRT     float64
AMT_REQ_CREDIT_BUREAU_YEAR    float64
Length: 122, dtype: object

In [16]:
df

SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,…,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
i64,i64,str,str,str,str,i64,f64,f64,f64,f64,str,str,str,str,str,f64,i64,i64,f64,i64,f64,i64,i64,i64,i64,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,…,f64,str,str,f64,str,str,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64
100002,1,"""Cash loans""","""M""","""N""","""Y""",0,202500.0,406597.5,24700.5,351000.0,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.018801,-9461,-637,-3648.0,-2120,null,1,1,0,1,1,0,"""Laborers""",1.0,2,2,"""WEDNESDAY""",10,0,0,0,…,0.0,"""reg oper account""","""block of flats""",0.0149,"""Stone, brick""","""No""",2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
100003,0,"""Cash loans""","""F""","""N""","""N""",0,270000.0,1293502.5,35698.5,1.1295e6,"""Family""","""State servant""","""Higher education""","""Married""","""House / apartment""",0.003541,-16765,-1188,-1186.0,-291,null,1,1,0,1,1,0,"""Core staff""",2.0,1,1,"""MONDAY""",11,0,0,0,…,0.01,"""reg oper account""","""block of flats""",0.0714,"""Block""","""No""",1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
100004,0,"""Revolving loans""","""M""","""Y""","""Y""",0,67500.0,135000.0,6750.0,135000.0,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,"""Laborers""",1.0,2,2,"""MONDAY""",9,0,0,0,…,null,null,null,null,null,null,0.0,0.0,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
100006,0,"""Cash loans""","""F""","""N""","""Y""",0,135000.0,312682.5,29686.5,297000.0,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Civil marriage""","""House / apartment""",0.008019,-19005,-3039,-9833.0,-2437,null,1,1,0,1,0,0,"""Laborers""",2.0,2,2,"""WEDNESDAY""",17,0,0,0,…,null,null,null,null,null,null,2.0,0.0,2.0,0.0,-617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null
100007,0,"""Cash loans""","""M""","""N""","""Y""",0,121500.0,513000.0,21865.5,513000.0,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.028663,-19932,-3038,-4311.0,-3458,null,1,1,0,1,0,0,"""Core staff""",1.0,2,2,"""THURSDAY""",11,0,0,0,…,null,null,null,null,null,null,0.0,0.0,0.0,0.0,-1106.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

### Clickstream

In [137]:
 "clickstream":{"uci_id":553,
        "name": "Clickstream Data for Online Retail",
        "repository_url": "https://archive.ics.uci.edu/dataset/553/clickstream+data+for+online+retail",
        "data_url": "https://archive.ics.uci.edu/static/public/553/clickstream+data+for+online+retail.data",
        'target_col': 'buy',
        'prediction_task': 'binary_classification',
        "shape": (287, 13),
    },

SyntaxError: illegal target for annotation (3019678334.py, line 1)

In [ ]:
clickstream_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["clickstream"]["uci_id"])
# target_col_name = "buy"
# prediction_task = "binary_classification"
X = pl.from_pandas(clickstream_data.data.features)
Y = pl.from_pandas(clickstream_data.data.targets)
clickstream = pl.concat([X, Y], how="horizontal")
clickstream

## UCI

In [ ]:
# NAME_URL_DICT_UCI = {
#     "bank-marketing": {
#         "uci_id": 222,
#         "name": "Bank Marketing",
#         "repository_url": "https://archive.ics.uci.edu/dataset/222/bank+marketing",
#         "data_url": "https://archive.ics.uci.edu/static/public/222/data.csv",
#         "target_col": "y",
#         "prediction_task": "binary_classification",
#         "shape": (45211, 17),
#         "types": {"intCols": 7, "floatCols": 10, "boolCols": 1, "catCols": 10},
#     },
#     "adult-census": {
#         "uci_id": 2,
#         "name": "Adult",
#         "repository_url": "https://archive.ics.uci.edu/dataset/2/adult",
#         "data_url": "https://archive.ics.uci.edu/static/public/2/adult.data",
#         "target_col": "income",
#         "prediction_task": "binary_classification",
#         "shape": (48842, 15),
#     },
#     "credit-defualt-taiwan": {
#         "uci_id": 350,
#         "name": "Default of Credit Card Clients",
#         "repository_url": "https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients",
#         "data_url": "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.data",
#         "target_col": "default payment next month",
#         "prediction_task": "binary_classification",
#         "shape": (30000, 24),
#     },
#     "cdc_diabetes": {
#         "uci_id": 891,
#         "name": "CDC Diabetes Health Indicators",
#         "repository_url": "https://archive.ics.uci.edu/dataset/891/diabetes",
#         "data_url": "https://archive.ics.uci.edu/static/public/891/diabetes.data",
#         "target_col": "Diabetes_binary",
#         "prediction_task": "binary_classification",
#         "shape": (253, 680, 21),
#     },
#     "census-income_full": {
#         "uci_id": 117,
#         "name": "Census Income (KDD)",
#         "repository_url": "https://archive.ics.uci.edu/dataset/117/census+income+kdd",
#         "data_url": "https://archive.ics.uci.edu/static/public/117/census+income+kdd.data",
#         "target_col": "income",
#         "prediction_task": "binary_classification",
#         "shape": (299285, 41),
#     },
#     "support2": {
#         "uci_id": 880,
#         "name": "SUPPORT2",
#         "repository_url": "https://archive.ics.uci.edu/dataset/880/support2",
#         "data_url": "https://archive.ics.uci.edu/static/public/880/support2.data",
#         "target_col": "death",
#         "prediction_task": "binary_classification",
#         "shape": (9105, 21),
#     },
#     "diabetes_130us": {
#         "uci_id": 296,
#         "name": "Diabetes 130-US hospitals for years 1999-2008",
#         "repository_url": "https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008",
#         "data_url": "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.data",
#         "target_col": "readmitted",
#         "prediction_task": "binary_classification",
#         "shape": (130, 50),
#         "types": {"numCols": 27, "catCols": 23},
#     },
#     "nursery": {
#         "uci_id": 76,
#         "name": "Nursery",
#         "repository_url": "https://archive.ics.uci.edu/dataset/76/nursery",
#         "data_url": "https://archive.ics.uci.edu/dataset/76/nursery.data",
#         "target_col": "class",
#         "prediction_task": "multi_class_classification",
#         "shape": (12960, 8),
#     },
#     "taiwanese_bankruptcy": {
#         "uci_id": 572,
#         "name": "Taiwanese Bankruptcy",
#         "repository_url": "https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction",
#         "data_url": "https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction.data",
#         "target_col": "Bankrupt?",
#         "prediction_task": "binary_classification",
#         "shape": (6819, 96),
#     },
#     "apartment_rent_classified": {
#         "uci_id": 555,
#         "name": "Apartment Rent Classified",
#         "repository_url": "https://archive.ics.uci.edu/dataset/555/apartment+for+rent+classified",
#         "data_url": "https://archive.ics.uci.edu/dataset/555/apartment+for+rent+classified.data",
#         "target_col": ["square_feet", "price"],
#         "prediction_task": "regression",
#         "shape": (99826, 21),
#     },
#     "covertype": {
#         "uci_id": 31,
#         "name": "Covertype",
#         "repository_url": "https://archive.ics.uci.edu/dataset/31/covertype",
#         "data_url": "https://archive.ics.uci.edu/dataset/31/covertype.data",
#         "target_col": "Cover_Type",
#         "prediction_task": "multi_class_classification",
#         "shape": (581012, 55),
#     },
#     # "adult": "https://archive.ics.uci.edu/static/public/2/adult.zip",
#     # "default": "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip",
#     # "magic": "https://archive.ics.uci.edu/static/public/159/magic+gamma+telescope.zip",
#     # "shoppers": "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
#     # "beijing": "https://archive.ics.uci.edu/static/public/381/beijing+pm2+5+data.zip",
#     # "news": "https://archive.ics.uci.edu/static/public/332/online+news+popularity.zip",
# }

### Bank-Marketing



In [4]:
bank_marketing_data = fetch_ucirepo(id=uci_dict["bank-marketing"]["uci_id"])
X = pl.from_pandas(bank_marketing_data.data.features)
Y = pl.from_pandas(bank_marketing_data.data.targets)
bank_marketing = pl.concat([X, Y], how="horizontal")
bank_marketing

age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
i64,str,str,str,str,i64,str,str,str,i64,str,i64,i64,i64,i64,str,str
58,"""management""","""married""","""tertiary""","""no""",2143,"""yes""","""no""",null,5,"""may""",261,1,-1,0,null,"""no"""
44,"""technician""","""single""","""secondary""","""no""",29,"""yes""","""no""",null,5,"""may""",151,1,-1,0,null,"""no"""
33,"""entrepreneur""","""married""","""secondary""","""no""",2,"""yes""","""yes""",null,5,"""may""",76,1,-1,0,null,"""no"""
47,"""blue-collar""","""married""",null,"""no""",1506,"""yes""","""no""",null,5,"""may""",92,1,-1,0,null,"""no"""
33,null,"""single""",null,"""no""",1,"""no""","""no""",null,5,"""may""",198,1,-1,0,null,"""no"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
51,"""technician""","""married""","""tertiary""","""no""",825,"""no""","""no""","""cellular""",17,"""nov""",977,3,-1,0,null,"""yes"""
71,"""retired""","""divorced""","""primary""","""no""",1729,"""no""","""no""","""cellular""",17,"""nov""",456,2,-1,0,null,"""yes"""
72,"""retired""","""married""","""secondary""","""no""",5715,"""no""","""no""","""cellular""",17,"""nov""",1127,5,184,3,"""success""","""yes"""


In [5]:
df_info = get_df_info_pl(bank_marketing)
# df_info

DataFrame shape: (45211, 17)
Number of numerical columns: 7: ['age', 'balance', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous']
  - Integer columns: 7: ['age', 'balance', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous']
  - Float columns: 0: []
Number of categorical columns: 10: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']
  - Binary columns: 4: ['default', 'housing', 'loan', 'y']
  - Boolean columns: 0: []
Total categorical cardinality: 46
Columns with missing values: 4


In [45]:
bank_marketing_data.metadata.keys()

dict_keys(['uci_id', 'name', 'repository_url', 'data_url', 'abstract', 'area', 'tasks', 'characteristics', 'num_instances', 'num_features', 'feature_types', 'demographics', 'target_col', 'index_col', 'has_missing_values', 'missing_values_symbol', 'year_of_dataset_creation', 'last_updated', 'dataset_doi', 'creators', 'intro_paper', 'additional_info'])

In [49]:
bank_marketing_data.metadata["feature_types"]
print(bank_marketing_data.metadata)

['Categorical', 'Integer']

### Adult

In [69]:
adult_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["adult"]["uci_id"])
target_col_name = "is_high_income"
prediction_task = "binary_classification"
X = pl.from_pandas(adult_data.data.features)
Y = pl.from_pandas(adult_data.data.targets)
adult = pl.concat([X, Y], how="horizontal")
adult = adult.with_columns(pl.when(pl.col("income") == ">50K.").then(1).otherwise(0).alias("is_high_income").cast(str))
adult = adult.drop(["income"])  # Drop un-used columns
adult

age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,is_high_income
i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str
39,"""State-gov""",77516,"""Bachelors""",13,"""Never-married""","""Adm-clerical""","""Not-in-family""","""White""","""Male""",2174,0,40,"""United-States""","""0"""
50,"""Self-emp-not-inc""",83311,"""Bachelors""",13,"""Married-civ-spouse""","""Exec-managerial""","""Husband""","""White""","""Male""",0,0,13,"""United-States""","""0"""
38,"""Private""",215646,"""HS-grad""",9,"""Divorced""","""Handlers-cleaners""","""Not-in-family""","""White""","""Male""",0,0,40,"""United-States""","""0"""
53,"""Private""",234721,"""11th""",7,"""Married-civ-spouse""","""Handlers-cleaners""","""Husband""","""Black""","""Male""",0,0,40,"""United-States""","""0"""
28,"""Private""",338409,"""Bachelors""",13,"""Married-civ-spouse""","""Prof-specialty""","""Wife""","""Black""","""Female""",0,0,40,"""Cuba""","""0"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
39,"""Private""",215419,"""Bachelors""",13,"""Divorced""","""Prof-specialty""","""Not-in-family""","""White""","""Female""",0,0,36,"""United-States""","""0"""
64,null,321403,"""HS-grad""",9,"""Widowed""",null,"""Other-relative""","""Black""","""Male""",0,0,40,"""United-States""","""0"""
38,"""Private""",374983,"""Bachelors""",13,"""Married-civ-spouse""","""Prof-specialty""","""Husband""","""White""","""Male""",0,0,50,"""United-States""","""0"""


In [71]:
df_info = get_df_info(adult)
# df_info

DataFrame shape: (48842, 15)
Number of numerical columns: 6
Number of categorical columns: 9
Total categorical cardinality: 107
Columns with missing values: 3


### Credit-default Taiwan


In [97]:
credit_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["credit-defualt"]["uci_id"])
# target_col_name = "is_default"
# prediction_task = "binary_classification"
X = pl.from_pandas(credit_data.data.features)
Y = pl.from_pandas(credit_data.data.targets)
credit = pl.concat([X, Y], how="horizontal")
# credit = credit.with_columns(
#             pl.when(pl.col("default payment next month") == 1).then(1).otherwise(0).alias("is_default").cast(str)
#         )
# credit = credit.drop(["default payment next month"])  # Drop un-used columns
credit


X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
220000,1,3,1,39,0,0,0,0,0,0,188948,192815,208365,88004,31237,15980,8500,20000,5003,3047,5000,1000,0
150000,1,3,2,43,-1,-1,-1,-1,0,0,1683,1828,3502,8979,5190,0,1837,3526,8998,129,0,0,0
30000,1,2,2,37,4,3,2,-1,0,0,3565,3356,2758,20878,20582,19357,0,0,22000,4200,2000,3100,1


In [85]:
# credit_data.metadata

In [84]:
credit_data.metadata["additional_info"]["variable_info"].split("\r\n")

['This research employed a binary variable, default payment (Yes = 1, No = 0), as the response variable. This study reviewed the literature and used the following 23 variables as explanatory variables:',
 'X1: Amount of the given credit (NT dollar): it includes both the individual consumer credit and his/her family (supplementary) credit.',
 'X2: Gender (1 = male; 2 = female).',
 'X3: Education (1 = graduate school; 2 = university; 3 = high school; 4 = others).',
 'X4: Marital status (1 = married; 2 = single; 3 = others).',
 'X5: Age (year).',
 'X6 - X11: History of past payment. We tracked the past monthly payment records (from April to September, 2005) as follows: X6 = the repayment status in September, 2005; X7 = the repayment status in August, 2005; . . .;X11 = the repayment status in April, 2005. The measurement scale for the repayment status is: -1 = pay duly; 1 = payment delay for one month; 2 = payment delay for two months; . . .; 8 = payment delay for eight months; 9 = payment

In [98]:
col_mapping = {
    "X1": "Credit_Amount",
    "X2": "Gender",
    "X3": "Education_Level",
    "X4": "Marital_Status",
    "X5": "Age",
    "X6": "Pay_0",
    "X7": "Pay_2",
    "X8": "Pay_3",
    "X9": "Pay_4",
    "X10": "Pay_5",
    "X11": "Pay_6",
    "X12": "Bill_Amount1",
    "X13": "Bill_Amount2",
    "X14": "Bill_Amount3",
    "X15": "Bill_Amount4",
    "X16": "Bill_Amount5",
    "X17": "Bill_Amount6",
    "X18": "Pay_Amount1",
    "X19": "Pay_Amount2",
    "X20": "Pay_Amount3",
    "X21": "Pay_Amount4",
    "X22": "Pay_Amount5",
    "X23": "Pay_Amount6",
    "Y": "default",
}

catCols = [
    "Gender",
    "Education_Level",
    "Marital_Status",
    "default",
]


credit = credit.rename(col_mapping)
credit = credit.with_columns(pl.col(catCols).cast(pl.Utf8))
credit

Credit_Amount,Gender,Education_Level,Marital_Status,Age,Pay_0,Pay_2,Pay_3,Pay_4,Pay_5,Pay_6,Bill_Amount1,Bill_Amount2,Bill_Amount3,Bill_Amount4,Bill_Amount5,Bill_Amount6,Pay_Amount1,Pay_Amount2,Pay_Amount3,Pay_Amount4,Pay_Amount5,Pay_Amount6,default
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str
20000,"""2""","""2""","""1""",24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,"""1"""
120000,"""2""","""2""","""2""",26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,"""1"""
90000,"""2""","""2""","""2""",34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,"""0"""
50000,"""2""","""2""","""1""",37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,"""0"""
50000,"""1""","""2""","""1""",57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,"""0"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
220000,"""1""","""3""","""1""",39,0,0,0,0,0,0,188948,192815,208365,88004,31237,15980,8500,20000,5003,3047,5000,1000,"""0"""
150000,"""1""","""3""","""2""",43,-1,-1,-1,-1,0,0,1683,1828,3502,8979,5190,0,1837,3526,8998,129,0,0,"""0"""
30000,"""1""","""2""","""2""",37,4,3,2,-1,0,0,3565,3356,2758,20878,20582,19357,0,0,22000,4200,2000,3100,"""1"""


### cdc_diabetes

In [102]:
cdc_diabetes_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["cdc_diabetes"]["uci_id"])
# target_col_name = "Diabetes_binary"
# prediction_task = "binary_classification"
X = pl.from_pandas(cdc_diabetes_data.data.features)
Y = pl.from_pandas(cdc_diabetes_data.data.targets)
cdc_diabetes = pl.concat([X, Y], how="horizontal")
cdc_diabetes

HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,1,1,40,1,0,0,0,0,1,0,1,0,5,18,15,1,0,9,4,3,0
0,0,0,25,1,0,0,1,0,0,0,0,1,3,0,0,0,0,7,6,1,0
1,1,1,28,0,0,0,0,1,0,0,1,1,5,30,30,1,0,9,4,8,0
1,0,1,27,0,0,0,1,1,1,0,1,0,2,0,0,0,0,11,3,6,0
1,1,1,24,0,0,0,1,1,1,0,1,0,2,3,0,0,0,11,5,4,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,1,1,45,0,0,0,0,1,1,0,1,0,3,0,5,0,1,5,6,7,0
1,1,1,18,0,0,0,0,0,0,0,1,0,4,0,0,1,0,11,2,4,1
0,0,1,28,0,0,0,1,1,0,0,1,0,1,0,0,0,0,2,5,2,0


In [ ]:
ordinalCols = ["GenHlth", "Education", "Income"]
intCols = ["MentHlth", "PhysHlth", "BMI", "Age"]
binaryCols = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "HeartDiseaseorAttack",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
    "Diabetes_binary",
]

In [ ]:
len(cdc_diabetes.columns) == len(ordinalCols) + len(intCols) + len(binaryCols)

### Census Income

probably shouldn't not include... 

In [114]:
census_income_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["census-income"]["uci_id"])
# target_col_name = "income"
# prediction_task = "binary_classification"
X = pl.from_pandas(census_income_data.data.features)
Y = pl.from_pandas(census_income_data.data.targets)
census_income = pl.concat([X, Y], how="horizontal")
census_income

AAGE,ACLSWKR,ADTINK,ADTOCC,AHGA,AHSCOL,AMARITL,AMJIND,AMJOCC,ARACE,AREORGN,ASEX,AUNMEM,AUNTYPE,AWKSTAT,CAPGAIN,GAPLOSS,DIVVAL,FILESTAT,GRINREG,GRINST,HHDFMX,HHDREL,MARSUPWRT,MIGMTR1,MIGMTR3,MIGMTR4,MIGSAME,MIGSUN,NOEMP,PARENT,PEFNTVTY,PEMNTVTY,PENATVTY,PRCITSHP,SEOTR,VETQVA,VETYN,WKSWORK,AHRSPAY,year,income
i64,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,str,str,str,str,str,f64,str,str,str,str,str,i64,str,str,str,str,str,i64,str,i64,i64,i64,i64,str
73,""" Not in universe""",0,0,""" High school graduate""",""" Not in universe""",""" Widowed""",""" Not in universe or children""",""" Not in universe""",""" White""",""" All other""",""" Female""",""" Not in universe""",""" Not in universe""",""" Not in labor force""",0,0,0,""" Nonfiler""",""" Not in universe""",""" Not in universe""",""" Other Rel 18+ ever marr not i…",""" Other relative of householder""",1700.09,null,null,null,""" Not in universe under 1 year …",null,0,""" Not in universe""",""" United-States""",""" United-States""",""" United-States""",""" Native- Born in the United St…",0,""" Not in universe""",2,0,0,95,"""-50000"""
58,""" Self-employed-not incorporate…",4,34,""" Some college but no degree""",""" Not in universe""",""" Divorced""",""" Construction""",""" Precision production craft & …",""" White""",""" All other""",""" Male""",""" Not in universe""",""" Not in universe""",""" Children or Armed Forces""",0,0,0,""" Head of household""",""" South""",""" Arkansas""",""" Householder""",""" Householder""",1053.55,""" MSA to MSA""",""" Same county""",""" Same county""",""" No""",""" Yes""",1,""" Not in universe""",""" United-States""",""" United-States""",""" United-States""",""" Native- Born in the United St…",0,""" Not in universe""",2,52,0,94,"""-50000"""
18,""" Not in universe""",0,0,""" 10th grade""",""" High school""",""" Never married""",""" Not in universe or children""",""" Not in universe""",""" Asian or Pacific Islander""",""" All other""",""" Female""",""" Not in universe""",""" Not in universe""",""" Not in labor force""",0,0,0,""" Nonfiler""",""" Not in universe""",""" Not in universe""",""" Child 18+ never marr Not in a…",""" Child 18 or older""",991.95,null,null,null,""" Not in universe under 1 year …",null,0,""" Not in universe""",""" Vietnam""",""" Vietnam""",""" Vietnam""",""" Foreign born- Not a citizen o…",0,""" Not in universe""",2,0,0,95,"""-50000"""
9,""" Not in universe""",0,0,""" Children""",""" Not in universe""",""" Never married""",""" Not in universe or children""",""" Not in universe""",""" White""",""" All other""",""" Female""",""" Not in universe""",""" Not in universe""",""" Children or Armed Forces""",0,0,0,""" Nonfiler""",""" Not in universe""",""" Not in universe""",""" Child <18 never marr not in s…",""" Child under 18 never married""",1758.14,""" Nonmover""",""" Nonmover""",""" Nonmover""",""" Yes""",""" Not in universe""",0,""" Both parents present""",""" United-States""",""" United-States""",""" United-States""",""" Native- Born in the United St…",0,""" Not in universe""",0,0,0,94,"""-50000"""
10,""" Not in universe""",0,0,""" Children""",""" Not in universe""",""" Never married""",""" Not in universe or children""",""" Not in universe""",""" White""",""" All other""",""" Female""",""" Not in universe""",""" Not in universe""",""" Children or Armed Forces""",0,0,0,""" Nonfiler""",""" Not in universe""",""" Not in universe""",""" Child <18 never marr not in s…",""" Child under 18 never married""",1069.16,""" Nonmover""",""" Nonmover""",""" Nonmover""",""" Yes""",""" Not in universe""",0,""" Both parents present""",""" United-States""",""" United-States""",""" United-States""",""" Native- Born in the United St…",0,""" Not in universe""",0,0,0,94,"""-50000"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
87,""" Not in universe""",0,0,""" 7th and 8th grade""",""" Not in universe""",""" Married-civilian spouse prese…",""" Not in universe or children""",""" Not in univers

### Support2

In [117]:
support2_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["support2"]["uci_id"])
# target_col_name = "income"
# prediction_task = "binary_classification"
X = pl.from_pandas(support2_data.data.features)
Y = pl.from_pandas(support2_data.data.targets)
support2 = pl.concat([X, Y], how="horizontal")
support2

age,sex,dzgroup,dzclass,num.co,edu,income,scoma,charges,totcst,totmcst,avtisst,race,sps,aps,surv2m,surv6m,hday,diabetes,dementia,ca,prg2m,prg6m,dnr,dnrday,meanbp,wblc,hrt,resp,temp,pafi,alb,bili,crea,sod,ph,glucose,bun,urine,adlp,adls,adlsc,death,hospdead,sfdm2
f64,str,str,str,i64,f64,str,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,i64,i64,i64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,str
62.84998,"""male""","""Lung Cancer""","""Cancer""",0,11.0,"""$11-$25k""",0.0,9715.0,null,null,7.0,"""other""",33.898438,20.0,0.262939,0.036995,1,0,0,"""metastatic""",0.5,0.25,"""no dnr""",5.0,97.0,6.0,69.0,22.0,36.0,388.0,1.7998047,0.199982,1.199951,141.0,7.459961,null,null,null,7.0,7.0,7.0,0,0,null
60.33899,"""female""","""Cirrhosis""","""COPD/CHF/Cirrhosis""",2,12.0,"""$11-$25k""",44.0,34496.0,null,null,29.0,"""white""",52.695312,74.0,0.001,0.0,3,0,0,"""no""",0.0,0.0,null,null,43.0,17.097656,112.0,34.0,34.59375,98.0,null,null,5.5,132.0,7.25,null,null,null,null,1.0,1.0,1,1,"""<2 mo. follow-up"""
52.74698,"""female""","""Cirrhosis""","""COPD/CHF/Cirrhosis""",2,12.0,"""under $11k""",0.0,41094.0,null,null,13.0,"""white""",20.5,45.0,0.790894,0.664917,4,0,0,"""no""",0.75,0.5,"""no dnr""",17.0,70.0,8.5,88.0,28.0,37.39844,231.65625,null,2.199707,2.0,134.0,7.459961,null,null,null,1.0,0.0,0.0,1,0,"""<2 mo. follow-up"""
42.38498,"""female""","""Lung Cancer""","""Cancer""",2,11.0,"""under $11k""",0.0,3075.0,null,null,7.0,"""white""",20.097656,19.0,0.698975,0.411987,1,0,0,"""metastatic""",0.9,0.5,"""no dnr""",3.0,75.0,9.099609,88.0,32.0,35.0,null,null,null,0.799927,139.0,null,null,null,null,0.0,0.0,0.0,1,0,"""no(M2 and SIP pres)"""
79.88495,"""female""","""ARF/MOSF w/Sepsis""","""ARF/MOSF""",1,null,null,26.0,50127.0,null,null,18.666656,"""white""",23.5,30.0,0.634888,0.532959,3,0,0,"""no""",0.9,0.9,"""no dnr""",16.0,59.0,13.5,112.0,20.0,37.89844,173.3125,null,null,0.799927,143.0,7.509766,null,null,null,null,2.0,2.0,0,0,"""no(M2 and SIP pres)"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
66.073,"""male""","""ARF/MOSF w/Sepsis""","""ARF/MOSF""",1,8.0,null,0.0,52870.0,34329.3125,32042.75,20.333328,"""white""",16.296875,22.0,0.852905,0.80188,13,0,0,"""no""",0.8,0.512,"""no dnr""",23.0,109.0,7.399414,104.0,22.0,35.69531,280.0,3.699707,0.399963,1.099854,131.0,7.459961,188.0,21.0,null,null,0.0,0.0,0,0,null
55.15399,"""female""","""Coma""","""Coma""",1,11.0,null,41.0,35377.0,23558.5,22131.0469,18.0,"""white""",25.796875,31.0,0.553955,0.485962,1,0,0,"""no""",0.5,0.5,"""no dnr""",29.0,43.0,null,0.0,8.0,38.59375,218.5,null,null,5.899414,135.0,7.289062,190.0,49.0,0.0,null,0.0,0.0,0,0,null
70.38196,"""male""","""ARF/MOSF w/Sepsis""","""ARF/MOSF""",1,null,null,0.0,46564.0,31409.0156,31131.25,23.0,"""white""",22.699219,39.0,0.741943,0.660889,18,0,0,"""no""",0.9,0.8,"""no dnr""",8.0,111.0,8.3984375,83.0,24.0,36.69531,180.0,null,0.399963,2.699707,139.0,7.379883,189.0,60.0,3900.0,null,null,2.5253906,0,0,null


### diabetes_130us

In [119]:
diabetes_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["diabetes_130us"]["uci_id"])
# target_col_name = "income"
# prediction_task = "binary_classification"
X = pl.from_pandas(diabetes_data.data.features)
Y = pl.from_pandas(diabetes_data.data.targets)
diabetes = pl.concat([X, Y], how="horizontal")
diabetes

/opt/miniconda3/envs/mixtabank/lib/python3.13/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
str,str,str,str,i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Caucasian""","""Female""","""[0-10)""",null,6,25,1,1,null,"""Pediatrics-Endocrinology""",41,0,1,0,0,0,"""250.83""",null,null,1,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""NO"""
"""Caucasian""","""Female""","""[10-20)""",null,1,1,7,3,null,null,59,0,18,0,0,0,"""276""","""250.01""","""255""",9,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""",""">30"""
"""AfricanAmerican""","""Female""","""[20-30)""",null,1,1,7,2,null,null,11,5,13,2,0,1,"""648""","""250""","""V27""",6,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""NO"""
"""Caucasian""","""Male""","""[30-40)""",null,1,1,7,2,null,null,44,1,16,0,0,0,"""8""","""250.43""","""403""",7,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""NO"""
"""Caucasian""","""Male""","""[40-50)""",null,1,1,7,1,null,null,51,0,8,0,0,0,"""197""","""157""","""250""",5,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""NO"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""AfricanAmerican""","""Male""","""[70-80)""",null,1,3,7,3,"""MC""",null,51,0,16,0,0,0,"""250.13""","""291""","""458""",9,null,""">8""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Down""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""",""">30"""
"""AfricanAmerican""","""Female""","""[80-90)""",null,1,4,5,5,"""MC""",null,33,3,18,0,0,1,"""560""","""276""","""787""",9,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""NO"""
"""Caucasian""","""Male""","""[70-80)""",null,1,1,7,1,"""MC""",null,53,0,9,1,0,0,"""38""","""590""","""296""",13,null,null,"""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Down""","""No""","""No""","""No""","""No""","""No""","""Ch""","""Yes""","""NO"""


In [120]:
diabetes.null_count().sum()

race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2273,0,0,98569,0,0,0,0,40256,49949,0,0,0,0,0,0,21,358,1423,0,96420,84748,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Nursery

In [123]:
nursery_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["nursery"]["uci_id"])
# target_col_name = "income"
# prediction_task = "binary_classification"
X = pl.from_pandas(nursery_data.data.features)
Y = pl.from_pandas(nursery_data.data.targets)
nursery = pl.concat([X, Y], how="horizontal")
nursery

parents,has_nurs,form,children,housing,finance,social,health,class
str,str,str,str,str,str,str,str,str
"""usual""","""proper""","""complete""","""1""","""convenient""","""convenient""","""nonprob""","""recommended""","""recommend"""
"""usual""","""proper""","""complete""","""1""","""convenient""","""convenient""","""nonprob""","""priority""","""priority"""
"""usual""","""proper""","""complete""","""1""","""convenient""","""convenient""","""nonprob""","""not_recom""","""not_recom"""
"""usual""","""proper""","""complete""","""1""","""convenient""","""convenient""","""slightly_prob""","""recommended""","""recommend"""
"""usual""","""proper""","""complete""","""1""","""convenient""","""convenient""","""slightly_prob""","""priority""","""priority"""
…,…,…,…,…,…,…,…,…
"""great_pret""","""very_crit""","""foster""","""more""","""critical""","""inconv""","""slightly_prob""","""priority""","""spec_prior"""
"""great_pret""","""very_crit""","""foster""","""more""","""critical""","""inconv""","""slightly_prob""","""not_recom""","""not_recom"""
"""great_pret""","""very_crit""","""foster""","""more""","""critical""","""inconv""","""problematic""","""recommended""","""spec_prior"""


In [133]:
# total cardinality:
# get_df_info(nursery)

### taiwanese_bankruptcy

In [139]:
bankruptcy_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["taiwanese_bankruptcy"]["uci_id"])
# target_col_name = "Bankrupt?"
# prediction_task = "binary_classification"
X = pl.from_pandas(bankruptcy_data.data.features)
Y = pl.from_pandas(bankruptcy_data.data.targets)
bankruptcy = pl.concat([X, Y], how="horizontal")
bankruptcy

ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,Continuous interest rate (after tax),Operating Expense Rate,Research and development expense rate,Cash flow rate,Interest-bearing debt interest rate,Tax rate (A),Net Value Per Share (B),Net Value Per Share (A),Net Value Per Share (C),Persistent EPS in the Last Four Seasons,Cash Flow Per Share,Revenue Per Share (Yuan ¥),Operating Profit Per Share (Yuan ¥),Per Share Net profit before tax (Yuan ¥),Realized Sales Gross Profit Growth Rate,Operating Profit Growth Rate,After-tax Net Profit Growth Rate,Regular Net Profit Growth Rate,Continuous Net Profit Growth Rate,Total Asset Growth Rate,Net Value Growth Rate,Total Asset Return Growth Rate Ratio,Cash Reinvestment %,Current Ratio,Quick Ratio,Interest Expense Ratio,Total debt/Total net worth,Debt ratio %,…,Current Liability to Assets,Operating Funds to Liability,Inventory/Working Capital,Inventory/Current Liability,Current Liabilities/Liability,Working Capital/Equity,Current Liabilities/Equity,Long-term Liability to Current Assets,Retained Earnings to Total Assets,Total income/Total expense,Total expense/Assets,Current Asset Turnover Rate,Quick Asset Turnover Rate,Working capitcal Turnover Rate,Cash Turnover Rate,Cash Flow to Sales,Fixed Assets to Assets,Current Liability to Liability,Current Liability to Equity,Equity to Long-term Liability,Cash Flow to Total Assets,Cash Flow to Liability,CFO to Assets,Cash Flow to Equity,Current Liability to Current Assets,Liability-Assets Flag,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability,Bankrupt?
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,i64
0.370594,0.424389,0.40575,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,0.780985,0.000126,0.0,0.458143,0.000725,0.0,0.14795,0.14795,0.14795,0.169141,0.311664,0.01756,0.095921,0.138736,0.022102,0.848195,0.688979,0.688979,0.217535,4.9800e9,0.000327,0.2631,0.363725,0.002259,0.001208,0.629951,0.021266,0.207576,…,0.147308,0.334015,0.27692,0.001036,0.676269,0.721275,0.339077,0.025592,0.903225,0.002022,0.064856,7.01e8,6.5500e9,0.593831,4.58e8,0.671568,0.424206,0.676269,0.339077,0.126549,0.637555,0.458609,0.520382,0.312905,0.11825,0,0.716845,0.009219,0.622879,0.601453,0.82789,0.290202,0.026601,0.56405,1,0.016469,1
0.464291,0.538214,0.51673,0.610235,0.610235,0.998946,0.79738,0.809301,0.303556,0.781506,0.00029,0.0,0.461867,0.000647,0.0,0.182251,0.182251,0.182251,0.208944,0.318137,0.021144,0.093722,0.169918,0.02208,0.848088,0.689693,0.689702,0.21762,6.1100e9,0.000443,0.264516,0.376709,0.006016,0.004039,0.635172,0.012502,0.171176,…,0.056963,0.341106,0.289642,0.00521,0.308589,0.731975,0.32974,0.023947,0.931065,0.002226,0.025516,0.000107,7.7000e9,0.593916,2.4900e9,0.67157,0.468828,0.308589,0.32974,0.120916,0.6411,0.459001,0.567101,0.314163,0.047775,0,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794,1
0.426071,0.499019,0.472295,0.60145,0.601364,0.998857,0.796403,0.808388,0.302035,0.780284,0.000236,2.55e7,0.458521,0.00079,0.0,0.177911,0.177911,0.193713,0.180581,0.307102,0.005944,0.092338,0.142803,0.02276,0.848094,0.689463,0.68947,0.217601,7.2800e9,0.000396,0.264184,0.368913,0.011543,0.005348,0.629631,0.021248,0.207516,…,0.098162,0.336731,0.277456,0.013879,0.446027,0.742729,0.334777,0.003715,0.909903,0.0020

In [141]:
# dict(zip(bankruptcy.columns, bankruptcy.dtypes))

### apartment_rent_classified

In [ ]:
# floatCols = ["bathrooms","latitude","longitude","time"]
# intCols = ["price","price_display","square_feet","bedrooms","latitude","longitude","time"]


In [166]:
apartment_rent_data["data"]["features"].shape

(99826, 21)

In [164]:
apartment_rent_data = fetch_ucirepo(id=NAME_URL_DICT_UCI["apartment_rent_classified"]["uci_id"])
# target_col_name = "Bankrupt?"
# prediction_task = "binary_classification"
# X = pl.from_pandas(apartment_rent_data.data.features)
# Y = pl.from_pandas(apartment_rent_data.data.targets)
# apartment_rent = pl.concat([X, Y], how="horizontal")
# apartment_rent

/opt/miniconda3/envs/mixtabank/lib/python3.13/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0,5,6,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


### Covertype



In [ ]:
covertype = fetch_ucirepo(id=31)

In [179]:
# covertype["data"]["features"].nunique()
# covertype["data"]["features"].dtypes